Investigate issues leading to Zprime->ZH->ZbbHtt samples hasving no events passing selections in tautau channel

In [1]:
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import particle
from particle import PDGID, Particle

In [8]:
tree = uproot.open("/grid_mnt/data__data.polcms/cms/cuisset/ZHbbtautau/jobs/Zprime_Zh_Zbbhtautau_M600/Step_6/Ntuple_0.root:Events", 
                          #filter_name=["luminosityBlock", "event", "GenPart_*"]
                          ).arrays()

In [31]:
zz_tree = uproot.open("/data_CMS/cms/vernazza/MCProduction/2023_11_14/MyPrivateGridpacks/"
                    "gg_X_ZZbbtautau_M600/Step_4/Ntuple_0_numEvent1000.root:Events", 
                          #filter_name=["luminosityBlock", "event", "GenPart_*"]
                          ).arrays()

In [16]:
tree.fields()

TypeError: 'list' object is not callable

In [17]:
tree[[key for key in tree.fields if key.startswith("Tau")]]

<Array [{Tau_chargedIso: [3.56], ...}, ...] type='1000 * {Tau_chargedIso: v...'>

In [65]:
filters = {
    "doubleTaus" : lambda ar : ak.num(ar.Tau_eta) >= 2,
    "tauIdVsMu" : lambda ar : ak.num((ar.Tau_idDeepTau2017v2p1VSmu >= 1)) >=2,
    "tauIdVsE" : lambda ar : ak.num((ar.Tau_idDeepTau2017v2p1VSe >= 2)) >=2,
    "tau_dz" : lambda ar : ak.num((ar.Tau_dz <= 0.2)) >=2,
    "tau_decayMode" : lambda ar : ak.num( (ar.Tau_decayMode == 0) | (ar.Tau_decayMode == 1) |(ar.Tau_decayMode == 10) |(ar.Tau_decayMode == 11)) >=2,
    "Tau_pt" : lambda ar : ak.num(ar.Tau_pt > 20.) >= 2
}

In [66]:
def applyFilters(ar):
    prev_count = len(ar)
    current_ar = ar
    for name, filter in filters.items():
        filter_array = filter(current_ar)
        print(f"{name} : running eff {ak.sum(filter_array)/prev_count}, global eff {ak.sum(filter_array)/len(ar)}")
        current_ar = current_ar[filter_array]
        prev_count = len(current_ar)
    return current_ar

In [67]:
filtered = applyFilters(tree)
None

doubleTaus : running eff 0.526, global eff 0.526
tauIdVsMu : running eff 1.0, global eff 0.526
tauIdVsE : running eff 1.0, global eff 0.526
tau_dz : running eff 1.0, global eff 0.526
tau_decayMode : running eff 1.0, global eff 0.526
Tau_pt : running eff 1.0, global eff 0.526


In [64]:
filtered.Tau_decayMode

<Array [[1, 1], [0, 0], [1, ...], ..., [1, 1], [1, 0]] type='526 * var * int32'>

In [68]:
applyFilters(zz_tree).Tau_decayMode

doubleTaus : running eff 0.604, global eff 0.604
tauIdVsMu : running eff 1.0, global eff 0.604
tauIdVsE : running eff 1.0, global eff 0.604
tau_dz : running eff 1.0, global eff 0.604
tau_decayMode : running eff 1.0, global eff 0.604
Tau_pt : running eff 1.0, global eff 0.604


<Array [[1, 5], [0, 10], [...], ..., [1, 1], [1, 1]] type='604 * var * int32'>

In [45]:
doubleTaus = filterDoubleTaus(tree)
doubleTaus_zz = filterDoubleTaus(zz_tree)
len(doubleTaus)

526

In [46]:
filterDeepTauId(doubleTaus)
len(filterDeepTauId(doubleTaus))

498